## Extracao - VSOL (Excel, SITE)

RASCUNHO - ainda nao testado em Databricks real. Ajustar apos a primeira execucao com dado de verdade.

Le o Excel bruto que o Power Automate pousou na pasta do SharePoint (`sharepoint_vsol_bruto_folder`),
filtra so as linhas do `projeto` atual (via `Proposta Comercial`) e grava essa fatia na Bronze
(`BRONZE/vsol/`). Nao faz join com Station nem de-para de parametro -- isso e feito no
`02_limpeza_vsol.ipynb`, igual a separacao que ja existe hoje entre `01_extracao_api` e `02_limpeza`.

**Configuracao do Job**: a task deste notebook precisa se chamar `fetch_from_aga_api` na definicao do
Job da VSOL (mesma task key que a Campo usa) -- e assim que `02_limpeza_vsol`/`03_validacoes`/
`04_envio_sharepoint` encontram o `output_filename` sem precisar de nenhuma mudanca de codigo neles.
Cada Run desta Job processa **um projeto so** (ver `docs/vsol-integracao.md`, secao 10.2) -- o Power
Automate chama `Run Now` duas vezes por arquivo novo, uma por projeto.

## Setup

In [ ]:
dbutils.library.restartPython()
!pip install --upgrade pip
!pip install openpyxl
!pip install pandas
!pip install python-dotenv
dbutils.library.restartPython()

In [ ]:
import re
import datetime as _dt
import pandas as pd
from pyspark.sql.types import StructType, StructField, StringType

from config import get_config
from sharepoint_connector import download_file_by_name

In [ ]:
# ------- SUMARIO DE EXECUCAO -------
execucao_steps = []

def log_step(etapa, df=None, status="Sucesso", observacoes="", registros_lidos=None, registros_escritos=None):
    n = df.count() if df is not None else 0
    execucao_steps.append({
        "etapa": etapa,
        "status": status,
        "registros_lidos": registros_lidos if registros_lidos is not None else n,
        "registros_escritos": registros_escritos if registros_escritos is not None else n,
        "flags": _collect_flags(df) if df is not None else "\u2014",
        "observacoes": observacoes,
    })

def _collect_flags(df, colunas=None):
    from pyspark.sql import functions as F
    flag_cols = colunas if colunas is not None else [c for c in df.columns if c.startswith("flag_")]
    triggered = []
    for col_name in flag_cols:
        count = df.filter(F.col(col_name).isNotNull() & (F.col(col_name) != "")).count()
        if count > 0:
            triggered.append(f"{col_name}({count})")
    return "; ".join(triggered) if triggered else "\u2014"

def persistir_log():
    from pyspark.sql import Row
    try:
        notebook_name = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
        _execution_log_path = project_path + "/execution_log"
        _batch_ts = _dt.datetime.now().isoformat()
        _log_rows = [
            Row(
                batch_ts=_batch_ts,
                projeto=projeto,
                notebook=notebook_name.split("/")[-1],
                ordem=i,
                etapa=s["etapa"],
                status=s["status"],
                registros_lidos=int(s.get("registros_lidos") or 0),
                registros_escritos=int(s.get("registros_escritos") or 0),
                flags=str(s.get("flags") or "\u2014"),
                observacoes=str(s.get("observacoes") or ""),
                ts_registro=_dt.datetime.now().isoformat(),
            )
            for i, s in enumerate(execucao_steps)
        ]
        _log_df = spark.createDataFrame(_log_rows)
        _log_df.write.format("delta").mode("append").option("mergeSchema", "true").save(_execution_log_path)
        print(f"Log persistido: {len(_log_rows)} steps -> {_execution_log_path}")
    except Exception as _log_err:
        print(f"Aviso: falha ao persistir log de execucao - {_log_err}")

## Projeto e arquivo

In [ ]:
dbutils.widgets.text("projeto", "")
dbutils.widgets.text("arquivo", "")

projeto = dbutils.widgets.get("projeto").strip()
arquivo = dbutils.widgets.get("arquivo").strip()

cfg = get_config(projeto)  # falha alto se `projeto` estiver vazio ou nao cadastrado

if not arquivo:
    raise ValueError(
        "Widget 'arquivo' vazio - informe o nome do arquivo pousado na pasta VSOL do SharePoint "
        "(cfg['sharepoint_vsol_bruto_folder'])."
    )

In [ ]:
project_name = projeto
project_path = "/mnt/wst/" + project_name

folder_bronze = project_path + "/BRONZE/"
vsol_path = folder_bronze + "vsol/"

# dbutils.fs.mkdirs(vsol_path)

### Download do Excel (pasta de dados brutos VSOL no SharePoint)

In [ ]:
local_path = f"/tmp/{arquivo}"

result = download_file_by_name(
    folder_path=cfg["sharepoint_vsol_bruto_folder"],
    filename=arquivo,
    local_path=local_path,
)

if result["status"] != "success":
    execucao_steps.append({
        "etapa": "Download do arquivo VSOL",
        "status": "Erro",
        "registros_lidos": 0,
        "registros_escritos": 0,
        "flags": "\u2014",
        "observacoes": f"Falha ao baixar '{arquivo}': {result['error']}",
    })
    persistir_log()
    raise RuntimeError(f"Falha ao baixar '{arquivo}' de '{cfg['sharepoint_vsol_bruto_folder']}': {result['error']}")

execucao_steps.append({
    "etapa": "Download do arquivo VSOL",
    "status": "Sucesso",
    "registros_lidos": 0,
    "registros_escritos": 0,
    "flags": "\u2014",
    "observacoes": f"'{arquivo}' baixado ({result['size_bytes']} bytes) para {local_path}",
})

### Leitura e filtro por projeto

Layout SITE: aba unica, achatada (uma linha por amostra x parametro). Ver `docs/vsol-integracao.md`
secoes 3.1 e 14 para o estudo completo desse formato.

In [ ]:
df_raw = pd.read_excel(local_path, sheet_name=0, dtype=str)
df_raw.columns = [c.strip() for c in df_raw.columns]

COL_PROPOSTA = "Proposta Comercial"
COL_IDENT = "Identificacao" if "Identificacao" in df_raw.columns else "Identificação"
COL_TIPO = "Tipo de Amostra"
COL_CODIGO = "Codigo da Amostra" if "Codigo da Amostra" in df_raw.columns else "Código da Amostra"
COL_DATA_COLETA = "Data de Coleta"
COL_DATA_RECEB = "Data de Recebimento"
COL_DATA_PUB = "Data de Publicacao" if "Data de Publicacao" in df_raw.columns else "Data de Publicação"
COL_SITUACAO = "Situacao da Amostra" if "Situacao da Amostra" in df_raw.columns else "Situação da Amostra"
COL_IDENT_ANALISE = "Identificacao da Analise" if "Identificacao da Analise" in df_raw.columns else "Identificação da Análise"
COL_RESULTADO = "Resultado da Analise" if "Resultado da Analise" in df_raw.columns else "Resultado da Análise"
COL_UNIDADE = "Unidade de Medida da Analise" if "Unidade de Medida da Analise" in df_raw.columns else "Unidade de Medida da Análise"

colunas_esperadas = [COL_PROPOSTA, COL_IDENT, COL_TIPO, COL_CODIGO, COL_DATA_COLETA, COL_DATA_RECEB,
                     COL_DATA_PUB, COL_SITUACAO, COL_IDENT_ANALISE, COL_RESULTADO, COL_UNIDADE]
faltando = [c for c in colunas_esperadas if c not in df_raw.columns]
if faltando:
    execucao_steps.append({
        "etapa": "Leitura do Excel",
        "status": "Erro",
        "registros_lidos": 0,
        "registros_escritos": 0,
        "flags": "\u2014",
        "observacoes": f"Colunas esperadas nao encontradas no arquivo: {faltando}. Layout pode ter mudado.",
    })
    persistir_log()
    raise RuntimeError(f"Colunas esperadas nao encontradas: {faltando}")

execucao_steps.append({
    "etapa": "Leitura do Excel",
    "status": "Sucesso",
    "registros_lidos": len(df_raw),
    "registros_escritos": len(df_raw),
    "flags": "\u2014",
    "observacoes": f"{len(df_raw)} linhas lidas de '{arquivo}'",
})

### Filtro de campanha (projeto atual)

In [ ]:
# Reaproveita cfg["campanha_api"] (ja cadastrado pra Campo) como substring de busca -- os valores
# observados em `Proposta Comercial` sempre comecam com esse mesmo texto
# (ex.: "Projeto_1233_IC_CDM_CODEMIN/GO" contem "Projeto_1233_IC_CDM").
campanha_busca = cfg["campanha_api"]

registros_antes_filtro = len(df_raw)
df_projeto = df_raw[df_raw[COL_PROPOSTA].str.contains(re.escape(campanha_busca), na=False)].copy()
registros_depois_filtro = len(df_projeto)

execucao_steps.append({
    "etapa": "Filtro de campanha",
    "status": "Sucesso" if registros_depois_filtro > 0 else "Aviso",
    "registros_lidos": registros_antes_filtro,
    "registros_escritos": registros_depois_filtro,
    "flags": "\u2014",
    "observacoes": (
        f"{registros_depois_filtro} de {registros_antes_filtro} linhas pertencem a '{campanha_busca}'"
        if registros_depois_filtro
        else f"Nenhuma linha do arquivo pertence a '{campanha_busca}' -- arquivo pode ser so do outro projeto"
    ),
})

if df_projeto.empty:
    persistir_log()
    dbutils.notebook.exit(f"Nenhuma linha de '{campanha_busca}' em '{arquivo}'")

### Filtro de amostras pendentes (sem resultado ainda)

`Situacao da Amostra == "Recebida"` = amostra recebida no lab mas ainda sem resultado publicado
(ver `docs/vsol-integracao.md`, secao 14). Nao sao erro, sao pendentes -- ficam de fora desta carga e
entram numa proxima execucao quando o resultado sair.

In [ ]:
registros_antes_pendente = len(df_projeto)
df_projeto = df_projeto[df_projeto[COL_SITUACAO] != "Recebida"].copy()
registros_pendentes = registros_antes_pendente - len(df_projeto)

execucao_steps.append({
    "etapa": "Filtro de amostras pendentes",
    "status": "Sucesso",
    "registros_lidos": registros_antes_pendente,
    "registros_escritos": len(df_projeto),
    "flags": "\u2014",
    "observacoes": f"{registros_pendentes} linha(s) 'Recebida' (sem resultado ainda) deixadas de fora desta carga",
})

if df_projeto.empty:
    persistir_log()
    dbutils.notebook.exit("Todas as linhas do projeto estavam pendentes (Situacao == Recebida)")

### Parsing do resultado (qualificador + valor)

~85% dos resultados chegam como texto com qualificador embutido (ex.: `"< 0,0030"`). Extrai o
qualificador (`<`/`>`) separado do valor. **Nao deriva `limiteQuantificacao`/LQ daqui** -- essa
informacao precisa vir do laboratorio (ver `docs/vsol-integracao.md`, secao 4).

In [ ]:
_PATTERN_QUALIFICADOR = re.compile(r"^([<>])\s*(.+)$")
_PATTERN_NUMERO = re.compile(r"^-?\d+([.,]\d+)?$")

def _parse_resultado(raw):
    # Retorna (qualifier, resultadoNumerico, resultadoTexto) -- numerico e texto sao mutuamente
    # exclusivos: preenche um ou outro, nunca os dois (bug corrigido em 2026-09-14: resultadoTexto
    # estava vindo preenchido mesmo quando o valor era um numero puro).
    if pd.isna(raw):
        return (None, None, None)
    texto = str(raw).strip()
    m = _PATTERN_QUALIFICADOR.match(texto)
    qualifier = m.group(1) if m else None
    valor = m.group(2).strip() if m else texto
    if _PATTERN_NUMERO.match(valor):
        return (qualifier, valor, None)  # numero (com ou sem qualificador) -> resultadoNumerico
    return (qualifier, None, texto)  # nao e numero -> resultadoTexto

_parsed = df_projeto[COL_RESULTADO].map(_parse_resultado)
df_projeto["qualifier"] = _parsed.map(lambda t: t[0])
df_projeto["resultadoNumerico"] = _parsed.map(lambda t: t[1])  # comma->dot e cast ficam pro 02_limpeza_vsol
df_projeto["resultadoTexto"] = _parsed.map(lambda t: t[2])

qtd_qualificado = df_projeto["qualifier"].notna().sum()
execucao_steps.append({
    "etapa": "Parsing de resultado",
    "status": "Sucesso",
    "registros_lidos": len(df_projeto),
    "registros_escritos": len(df_projeto),
    "flags": "\u2014",
    "observacoes": f"{qtd_qualificado} de {len(df_projeto)} resultados com qualificador (<, >)",
})

### Padronizacao para o schema canonico (Bronze)

Mesmos nomes de campo que `01_extracao_api` ja produz (`parse_hga_api_json_to_df`), pra
`02_limpeza_vsol`/`03_validacoes`/`04_envio_sharepoint` nao precisarem saber a origem do dado.
`dataEnvioLab` e `dataHoraAnalise` ficam como colunas vazias de proposito -- nao existem no SITE
(bloqueado, depende do laboratorio, ver secao 4 do doc) mas o `02_limpeza_vsol` referencia essas
colunas pelo nome durante a normalizacao de data, entao precisam existir (mesmo vazias).

In [ ]:
df_bronze = pd.DataFrame({
    "campanha": df_projeto[COL_PROPOSTA],
    "idAmostra": df_projeto[COL_CODIGO].astype(str),
    "nomeAmostra": df_projeto[COL_IDENT],
    "descricaoAmostra": df_projeto[COL_IDENT],
    "matriz": df_projeto[COL_TIPO],
    "dataHoraAmostragem": df_projeto[COL_DATA_COLETA],
    "dataRecebLab": df_projeto[COL_DATA_RECEB],
    "dataEnvioLab": None,       # nao existe no SITE
    "dataHoraAnalise": None,    # nao existe no SITE -- pendente do laboratorio
    "dataLiberacao": df_projeto[COL_DATA_PUB],
    "laboratorio": "VSOL",      # nao existe coluna de lab no SITE -- assumido conforme decisao do Sinderley
    "parametroOriginal": df_projeto[COL_IDENT_ANALISE],
    "resultadoOriginal": df_projeto[COL_RESULTADO],
    "resultadoNumerico": df_projeto["resultadoNumerico"],
    "resultadoTexto": df_projeto["resultadoTexto"],
    "qualifier": df_projeto["qualifier"],
    "unidadeOriginal": df_projeto[COL_UNIDADE],
    "OrigemArquivo": "VSOL_SITE",
})

# Todas as colunas como string, igual ao schema que 01_extracao_api ja usa (StructType de StringType) --
# 02_limpeza_vsol faz os casts (data, numero) que forem necessarios.
df_bronze = df_bronze.astype(object).where(pd.notnull(df_bronze), None)
df_bronze = df_bronze.astype(str).where(df_bronze.notnull(), None)

### Gravacao Bronze (Delta)

In [ ]:
schema = StructType([StructField(c, StringType(), True) for c in df_bronze.columns])
df_spark = spark.createDataFrame(df_bronze, schema=schema)

nome_pasta = f"vsolSITE_{_dt.datetime.now().strftime('%Y%m%d_%H%M')}"
caminho_completo = f"{vsol_path}{nome_pasta}"

try:
    df_spark.write.format("delta").mode("overwrite").save(caminho_completo)
    execucao_steps.append({
        "etapa": "Gravacao Bronze",
        "status": "Sucesso",
        "registros_lidos": df_spark.count(),
        "registros_escritos": df_spark.count(),
        "flags": "\u2014",
        "observacoes": f"Salvo em {caminho_completo}",
    })
except Exception as _e:
    execucao_steps.append({
        "etapa": "Gravacao Bronze",
        "status": "Erro",
        "registros_lidos": df_spark.count(),
        "registros_escritos": 0,
        "flags": "\u2014",
        "observacoes": str(_e),
    })
    persistir_log()
    raise

print("Nome da pasta de salvamento:", nome_pasta)

In [ ]:
display(pd.DataFrame(execucao_steps))

In [ ]:
persistir_log()

In [ ]:
dbutils.jobs.taskValues.set(
    key="output_filename",
    value=nome_pasta,
)